# VCuboidFIT — đêm 14/10: seed PointPillars trên trainval + chẩn đoán V (planning/15)
Runtime **GPU A100** (Runtime → Change runtime type → A100, High-RAM). Chạy **Runtime → Run all**.

- Train PointPillars **chỉ** trên S của pool + 70 scene TRAIN chính thức ngoài pool (gói `seed_pp_v1`); **không bao giờ** V/T.
- Suy luận trên toàn job pool (nhãn không được đọc), chạy `rare_v24` trên **V** ở 3 cấu hình, chấm **chẩn đoán trên V**.
- **T đã mở một lần (pitch-v2, 13–14/10) — notebook này KHÔNG đụng T.** Kết quả V là chẩn đoán, không thay prereg.
- Không token, không API key, không VLM. Checkpoint lưu Drive riêng của bạn — không chia sẻ công khai.
- Ngắt runtime giữa chừng: kết nối lại rồi **Run all** lần nữa với cùng `RUN_ID` — ô nào đã xong trên Drive sẽ bỏ qua,
  train tiếp từ checkpoint đã đồng bộ (`SAVE/train/`).

In [ ]:
# @title 1 · Tham số, Drive, GPU (kernel chỉ dùng thư viện chuẩn; việc nặng chạy bằng subprocess)
RUN_ID = 'pp-trainval-20261014'   # @param {type:"string"} — giữ nguyên để resume cùng thư mục Drive
EPOCHS = 20                       # @param {type:"integer"}
BATCH = 16                        # @param {type:"integer"} — OOM thì ô 10 tự hạ 8 rồi 4
WORKERS = 8                       # @param {type:"integer"}
SWEEPS = 1                        # gói pool + seed KHÔNG có sweeps (SPEC-P01 §2) — đừng đổi
RUN_A2_MULTI_CLUSTER = False      # @param {type:"boolean"} — cấu hình phụ cluster_only multisensor (+~35 phút)
REQUIRE_A100 = True               # @param {type:"boolean"}
SYNC_EVERY_S = 300                # chu kỳ đồng bộ checkpoint lên Drive

import os, sys, json, time, hashlib, zipfile, shutil, subprocess, threading, re
from pathlib import Path
from datetime import datetime, timezone

T_NOTEBOOK = time.time()
def stamp(msg):
    print(f"[{datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')}Z] {msg}", flush=True)

from google.colab import drive
drive.mount('/content/drive')
MYDRIVE = Path('/content/drive/MyDrive')
def first_dir(cands, what):
    for c in cands:
        if (c / 'manifest.json').is_file():
            return c
    raise FileNotFoundError(f'Không thấy {what}: cần manifest.json ở một trong ' + ', '.join(map(str, cands)))
POOL_DIR = first_dir([MYDRIVE/'datatest/pool_v24', MYDRIVE/'vcf_pack/pool_v24', MYDRIVE/'pool_v24'], 'gói pool_v24')
SEED_DIR = first_dir([MYDRIVE/'datatest/seed_pp_v1', MYDRIVE/'vcf_pack/seed_pp_v1', MYDRIVE/'seed_pp_v1'], 'gói seed_pp_v1')
RESULTS = MYDRIVE / 'vcf_results'
SAVE = RESULTS / RUN_ID
for sub in ('steps', 'logs', 'seed', 'train', 't1', 'runs'):
    (SAVE / sub).mkdir(parents=True, exist_ok=True)
RUN = Path('/content/vcf_night')          # đường cố định: OpenPCDet ghi đường tuyệt đối vào thư mục output
REPO, VCF_HOME = RUN / 'repo', RUN / 'vcf_home'
for d in (RUN, RUN / 'logs', RUN / 'markers'):
    d.mkdir(parents=True, exist_ok=True)
PY = sys.executable

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError('Runtime không có GPU: Runtime → Change runtime type → A100')
GPU = gpu.stdout.strip()
if REQUIRE_A100 and 'A100' not in GPU:
    raise RuntimeError(f'Cần A100, đang có {GPU}. Đổi runtime hoặc đặt REQUIRE_A100=False (ước tính thời gian không còn đúng)')
mem_gb = int(open('/proc/meminfo').read().split('MemTotal:')[1].split()[0]) / 1e6
free_gb = shutil.disk_usage('/content').free / 1e9
stamp(f'GPU {GPU} · RAM {mem_gb:.0f} GB · đĩa trống {free_gb:.0f} GB')
if free_gb < 60:
    raise RuntimeError(f'Đĩa /content còn {free_gb:.0f} GB < 60 GB (pool 13,3 + seed 2,0 + job + cache)')
if mem_gb < 40:
    print('CẢNH BÁO: RAM < 40 GB — bật High-RAM; devkit trainval + train song song có thể hết RAM')
stamp(f'RUN_ID={RUN_ID} · POOL_DIR={POOL_DIR} · SEED_DIR={SEED_DIR} · SAVE={SAVE}')

In [ ]:
# @title 2 · Hàm đo (thời gian, VRAM đỉnh), lệnh có log, đánh dấu bước trên Drive
assert 'SAVE' in globals(), 'Chạy ô 1 trước'
TIMING_PATH = SAVE / 'timing.json'
TIMING = json.loads(TIMING_PATH.read_text(encoding='utf-8'))['steps'] if TIMING_PATH.is_file() else []

def gpu_used_mib():
    try:
        out = subprocess.check_output(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'], text=True)
        return max(float(v) for v in out.split())
    except Exception:
        return None

def save_timing():
    tmp = TIMING_PATH.with_suffix('.part')
    tmp.write_text(json.dumps(dict(run_id=RUN_ID, gpu=GPU, steps=TIMING), indent=2), encoding='utf-8')
    os.replace(tmp, TIMING_PATH)

def measured(name, action):
    stop, samples = threading.Event(), []
    def monitor():
        while not stop.is_set():
            v = gpu_used_mib()
            if v is not None:
                samples.append(v)
            stop.wait(1.0)
    th = threading.Thread(target=monitor, daemon=True); th.start()
    t0, status = time.perf_counter(), 0
    stamp(f'▶ {name}')
    try:
        return action()
    except BaseException as exc:
        status = getattr(exc, 'returncode', 1)
        raise
    finally:
        stop.set(); th.join()
        TIMING.append(dict(step=name, elapsed_s=round(time.perf_counter() - t0, 1), exit_code=status,
                           gpu_peak_used_mib=max(samples) if samples else None,
                           gpu_measurement='nvidia-smi device-wide 1 s (gồm cả train nền nếu đang chạy)',
                           utc=datetime.now(timezone.utc).isoformat()))
        save_timing()
        stamp(f"◀ {name}: {TIMING[-1]['elapsed_s']} s · exit {status} · VRAM đỉnh {TIMING[-1]['gpu_peak_used_mib']} MiB")

def env():
    e = os.environ.copy()
    e.update(PYTHONIOENCODING='utf-8', PYTHON=PY, VCF_HOME=str(VCF_HOME), PCDET_ROOT=str(VCF_HOME / 'OpenPCDet'),
             PYTHONPATH=f"{REPO / 'model/worker'}:{VCF_HOME / 'OpenPCDet'}")
    return e

def command(name, args, cwd=None, quiet=False):
    # Chạy lệnh, in log trực tiếp + ghi RUN/logs/<name>.log, chép log lên Drive; exit != 0 => lỗi
    log = RUN / 'logs' / f'{name}.log'
    def run():
        with log.open('a', encoding='utf-8') as f:
            f.write(f'\n# {datetime.now(timezone.utc).isoformat()} $ ' + ' '.join(map(str, args)) + '\n')
            p = subprocess.Popen(list(map(str, args)), cwd=str(cwd or REPO / 'model/worker'), env=env(),
                                 stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace')
            last = []
            for line in p.stdout:
                f.write(line)
                last = (last + [line])[-40:]
                if not quiet:
                    print(line, end='')
            p.wait()
        shutil.copy2(log, SAVE / 'logs' / log.name)
        if p.returncode != 0:
            if quiet:
                print(''.join(last))
            raise subprocess.CalledProcessError(p.returncode, args)
        return ''.join(last)
    return measured(name, run)

def digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            h.update(chunk)
    return h.hexdigest()

def done(name, **info):
    (SAVE / 'steps' / f'{name}.json').write_text(json.dumps(dict(utc=datetime.now(timezone.utc).isoformat(), **info),
                                                            indent=2, default=str), encoding='utf-8')
def is_done(name):
    return (SAVE / 'steps' / f'{name}.json').is_file()
def local_ok(name):
    return (RUN / 'markers' / name).is_file()
def mark_local(name):
    (RUN / 'markers' / name).write_text(datetime.now(timezone.utc).isoformat())

def to_drive(src, dst):
    # chép một file lên Drive nguyên tử (.part rồi đổi tên) + ghi <dst>.sha256
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.part')
    shutil.copy2(src, tmp); os.replace(tmp, dst)
    sha = digest(dst)
    dst.with_name(dst.name + '.sha256').write_text(f'{sha}  {dst.name}\n')
    return sha

def verified(dst):
    s = dst.with_name(dst.name + '.sha256')
    return dst.is_file() and s.is_file() and s.read_text().split()[0] == digest(dst)

def extract_checked(zpath, target):
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath) as z:
        for item in z.infolist():
            dest = (target / item.filename).resolve()
            if not dest.is_relative_to(target.resolve()) or (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('ZIP entry không an toàn: ' + item.filename)
        z.extractall(target)
stamp(f'helpers OK · {len(TIMING)} bước đã đo từ lần trước')

In [ ]:
# @title 3 · Code: ZIP HEAD từ gói seed, kiểm sha256 + commit trong comment ZIP
assert 'measured' in globals(), 'Chạy ô 2 trước'
SEED_MANIFEST = json.loads((SEED_DIR / 'manifest.json').read_text(encoding='utf-8-sig'))
assert SEED_MANIFEST.get('format') == 'vcf-seed-colab-v1', 'manifest seed sai định dạng'
REPO_SHA = SEED_MANIFEST.get('repo_sha')
assert REPO_SHA and 'code' in SEED_MANIFEST, 'gói seed thiếu code ZIP: đóng gói lại với --code'
CODE_ZIP = SEED_DIR / SEED_MANIFEST['code']['name']
mark = REPO / '.extracted_sha'
if mark.is_file() and mark.read_text().strip() == REPO_SHA:
    stamp(f'code {REPO_SHA[:7]} đã giải nén — bỏ qua')
else:
    def unpack_code():
        assert digest(CODE_ZIP) == SEED_MANIFEST['code']['sha256'], 'sha256 code ZIP lệch manifest'
        with zipfile.ZipFile(CODE_ZIP) as z:
            assert z.comment.decode().strip() == REPO_SHA, 'commit trong ZIP lệch manifest'
        shutil.rmtree(REPO, ignore_errors=True)
        extract_checked(CODE_ZIP, REPO)
        mark.write_text(REPO_SHA)
    measured('code_extract', unpack_code)
SCRIPT = REPO / 'model/scripts/colab_pp_night.py'
assert SCRIPT.is_file() and (REPO / 'model/worker/c4/lidar/tier1/train_seed.py').is_file()
done('code', repo_sha=REPO_SHA, code_sha256=SEED_MANIFEST['code']['sha256'])
stamp(f'CODE OK {REPO_SHA}')

In [ ]:
# @title 4 · Cài môi trường: worker[gpu] + OpenPCDet (colab_setup.sh) + trọng số OpenCLIP (~10–20 phút lần đầu)
assert 'REPO_SHA' in globals(), 'Chạy ô 3 trước'
INSTALL_MARK = Path(f'/tmp/.vcf_night_install_{REPO_SHA[:12]}')   # /tmp: phiên mới phải cài lại
if INSTALL_MARK.is_file():
    stamp('môi trường đã cài trong runtime này — bỏ qua')
else:
    command('install_worker_gpu', [PY, '-m', 'pip', 'install', '-q', '-e', str(REPO / 'model/worker') + '[gpu]'], quiet=True)
    command('colab_setup', ['bash', REPO / 'model/scripts/colab_setup.sh'], cwd=REPO)
    command('fetch_weights', [PY, REPO / 'model/scripts/fetch_weights.py', '--dir', REPO / 'model/workspace/weights'], quiet=True)
    INSTALL_MARK.write_text(REPO_SHA)
probe = ("import numpy, torch, spconv, open_clip; import pcdet.ops.iou3d_nms.iou3d_nms_cuda; "
         "print('numpy', numpy.__version__, '| torch', torch.__version__, torch.version.cuda, '| cuda', torch.cuda.is_available(), "
         "'| spconv', spconv.__version__, '| open_clip', open_clip.__version__)")
command('env_probe', [PY, '-c', probe])
train_py = (VCF_HOME / 'OpenPCDet/tools/train.py').read_text(encoding='utf-8')
RESUME_SUPPORTED = 'ckpt_list' in train_py
stamp(f"OpenPCDet train.py tự resume từ ckpt: {RESUME_SUPPORTED} (False ⇒ ngắt runtime = train lại từ đầu)")
done('install', resume_supported=RESUME_SUPPORTED)

In [ ]:
# @title 5 · Dữ liệu: giải nén gói pool (170 scene, không sweeps) + gói seed (70 scene TRAIN ngoài pool) — kiểm sha từng part
assert 'RESUME_SUPPORTED' in globals(), 'Chạy ô 4 trước'
POOL_MANIFEST = json.loads((POOL_DIR / 'manifest.json').read_text(encoding='utf-8-sig'))

def unpack_parts(mdir, manifest, target):
    n = 0
    for part in manifest['parts']:
        archive = mdir / part['name']
        assert Path(part['name']).name == part['name'], 'tên part lạ'
        assert archive.stat().st_size == part['size_bytes'], part['name'] + ' sai kích thước'
        assert digest(archive) == part['sha256'], part['name'] + ' SHA256 lệch'
        with zipfile.ZipFile(archive) as z:
            cnt = sum(not f.is_dir() for f in z.infolist())
        assert cnt == part['file_count'], part['name'] + ' số file lệch'
        extract_checked(archive, target)
        n += cnt
        stamp(f"  {part['name']} OK ({cnt} file)")
    assert n == manifest['file_count'], f"tổng file {n} != {manifest['file_count']}"
    return target / 'data'

POOL_ROOT, SEED_ROOT = RUN / 'pool_data' / 'data', RUN / 'seed_data' / 'data'
if local_ok('pool_unpacked'):
    stamp('pool đã giải nén — bỏ qua')
else:
    assert POOL_MANIFEST['keyframes'] == 6753 and POOL_MANIFEST['camera_files'] == 40518
    shutil.rmtree(RUN / 'pool_data', ignore_errors=True)
    measured('pool_unpack', lambda: unpack_parts(POOL_DIR, POOL_MANIFEST, RUN / 'pool_data'))
    assert (POOL_ROOT / 'v1.0-trainval/scene.json').is_file() and (POOL_ROOT / 'maps').is_dir()
    assert not (POOL_ROOT / 'sweeps').exists(), 'gói pool không được có sweeps'
    mark_local('pool_unpacked')
if local_ok('seed_unpacked'):
    stamp('seed đã giải nén — bỏ qua')
else:
    assert SEED_MANIFEST['sweeps'] == 0 and SEED_MANIFEST['seed_scenes'] == len(SEED_MANIFEST['scenes'])
    shutil.rmtree(RUN / 'seed_data', ignore_errors=True)
    measured('seed_unpack', lambda: unpack_parts(SEED_DIR, SEED_MANIFEST, RUN / 'seed_data'))
    assert not any(SEED_ROOT.glob('v1.0-*')), 'gói seed không được kèm bảng meta (dùng meta của pool)'
    mark_local('seed_unpacked')
n_lidar = sum(1 for _ in (POOL_ROOT / 'samples/LIDAR_TOP').iterdir())
n_seed = sum(1 for _ in (SEED_ROOT / 'samples/LIDAR_TOP').iterdir() if not _.is_symlink())
stamp(f'DATA OK · pool LIDAR keyframe {n_lidar} · seed LIDAR keyframe {n_seed} (manifest {SEED_MANIFEST["keyframes"]})')
assert n_lidar == 6753 and n_seed == SEED_MANIFEST['keyframes']

In [ ]:
# @title 6 · Chia seed: S = pool S (7 scene) + 70 scene gói seed; holdout P cho eval OpenPCDet; CẤM V/T
assert 'SEED_ROOT' in globals(), 'Chạy ô 5 trước'
EXP = RUN / 'exp'
out = command('seed_split', [PY, SCRIPT, 'seed-split', '--pool-root', POOL_ROOT, '--seed-root', SEED_ROOT,
                             '--exp', EXP, '--manifest', SEED_DIR / 'manifest.json', '--link'])
SEED_SPLIT = json.loads((EXP / 'seed_split.json').read_text(encoding='utf-8'))
assert SEED_SPLIT['pool_split_sha256'] == SEED_MANIFEST['pool_split_sha256']
assert SEED_SPLIT['train_from_seed_pack'] == SEED_MANIFEST['keyframes']
prev = SAVE / 'seed' / 'seed_split.json'
if prev.is_file():
    old = json.loads(prev.read_text(encoding='utf-8'))
    assert old['index_sha256'] == SEED_SPLIT['index_sha256'], 'tập train khác lần trước cùng RUN_ID: đổi RUN_ID'
to_drive(EXP / 'index.parquet', SAVE / 'seed' / 'exp_index.parquet')
to_drive(EXP / 'seed_split.json', SAVE / 'seed' / 'seed_split.json')
done('seed_split', **SEED_SPLIT)
stamp(f"SEED SPLIT OK · train {SEED_SPLIT['train_frames']} frame / {SEED_SPLIT['train_scenes']} scene · "
      f"holdout P {SEED_SPLIT['holdout_P_frames']} frame")

In [ ]:
# @title 7 · Train seed PointPillars CHẠY NỀN (không chặn) + đồng bộ checkpoint lên Drive mỗi SYNC_EVERY_S giây
assert 'SEED_SPLIT' in globals(), 'Chạy ô 6 trước'
SEED_SAVE, TRAIN_SAVE, PCDET_OUT = SAVE / 'seed', SAVE / 'train' / 'pcdet_output', EXP / 't1' / 'pcdet_output'
TRAIN_LOG = RUN / 'logs' / 'train.log'
SEED_FILES = {'ckpt/seed_latest.pth': 'seed_latest.pth', 'cfg/pp_seed.yaml': 'pp_seed.yaml',
              'train_config.yaml': 'train_config.yaml', 'seed_tokens.txt': 'seed_tokens.txt'}

def seed_on_drive():
    return all(verified(SEED_SAVE / d) for d in SEED_FILES.values())

def restore_seed():
    for rel, d in SEED_FILES.items():
        dst = EXP / 't1' / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.is_file():
            shutil.copy2(SEED_SAVE / d, dst)

def mirror_ckpts(final=False):
    # chỉ chép ckpt đã ổn định (>60 s không đổi) ; xoá bản Drive không còn ở local (OpenPCDet giữ 2 bản)
    if not PCDET_OUT.is_dir():
        return 0
    n, now, keep = 0, time.time(), set()
    for f in PCDET_OUT.rglob('*.pth'):
        rel = f.relative_to(PCDET_OUT); keep.add(rel)
        dst = TRAIN_SAVE / rel
        if (final or now - f.stat().st_mtime > 60) and (not dst.is_file() or dst.stat().st_size != f.stat().st_size
                                                        or dst.stat().st_mtime < f.stat().st_mtime):
            dst.parent.mkdir(parents=True, exist_ok=True)
            tmp = dst.with_name(dst.name + '.part'); shutil.copy2(f, tmp); os.replace(tmp, dst); n += 1
    for g in TRAIN_SAVE.rglob('*.pth') if TRAIN_SAVE.is_dir() else []:
        if g.relative_to(TRAIN_SAVE) not in keep and keep:
            g.unlink()
    if TRAIN_LOG.is_file():
        shutil.copy2(TRAIN_LOG, SAVE / 'logs' / 'train.log')
    return n

def restore_ckpts():
    n = 0
    for g in TRAIN_SAVE.rglob('*.pth') if TRAIN_SAVE.is_dir() else []:
        dst = PCDET_OUT / g.relative_to(TRAIN_SAVE)
        if not dst.is_file():
            dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(g, dst); n += 1
    return n

def train_running():
    return subprocess.run(['pgrep', '-f', 'c4.lidar.tier1.train_seed'], capture_output=True).returncode == 0

def start_train(batch):
    global TRAIN_PROC, TRAIN_T0, TRAIN_BATCH
    TRAIN_BATCH, TRAIN_T0 = batch, time.time()
    f = TRAIN_LOG.open('a', encoding='utf-8')
    args = [PY, '-m', 'c4.lidar.tier1.train_seed', '--exp', EXP, '--nusc', SEED_ROOT, '--sweeps', SWEEPS,
            '--epochs', EPOCHS, '--batch', batch, '--workers', WORKERS]
    f.write(f'\n# {datetime.now(timezone.utc).isoformat()} $ ' + ' '.join(map(str, args)) + '\n'); f.flush()
    TRAIN_PROC = subprocess.Popen(list(map(str, args)), cwd=str(REPO / 'model/worker'), env=env(), stdout=f,
                                  stderr=subprocess.STDOUT)
    stamp(f'train nền PID {TRAIN_PROC.pid} · batch {batch} · epochs {EPOCHS} · sweeps {SWEEPS} · log {TRAIN_LOG}')

if globals().get('SYNC_STOP') is not None:   # chạy lại ô: dừng luồng đồng bộ cũ
    globals()['SYNC_STOP'].set()
SYNC_STOP = threading.Event()
def sync_loop():
    while not SYNC_STOP.wait(SYNC_EVERY_S):
        try:
            k = mirror_ckpts()
            if k:
                stamp(f'đồng bộ {k} checkpoint lên Drive')
        except Exception as e:
            print('sync lỗi (thử lại chu kỳ sau):', e)

TRAIN_PROC, TRAIN_BATCH, TRAIN_T0 = None, BATCH, time.time()
if seed_on_drive():
    restore_seed()
    stamp('đã có checkpoint seed cuối trên Drive (sha khớp) — BỎ QUA train')
else:
    if train_running():
        stamp('train_seed đang chạy (từ lần chạy ô trước) — không khởi động lại; ô 10 sẽ chờ')
    else:
        k = restore_ckpts()
        if k:
            stamp(f'khôi phục {k} checkpoint giữa chừng từ Drive ⇒ OpenPCDet train tiếp (resume={RESUME_SUPPORTED})')
        start_train(BATCH)
    threading.Thread(target=sync_loop, daemon=True).start()

In [ ]:
# @title 8 · Chuẩn bị job pool (CPU + CLIP GPU, chạy song song với train): index, chia, Tầng 0, CLIP, bối cảnh (fit V)
assert 'EXP' in globals(), 'Chạy ô 6–7 trước'
POOL_JOB = RUN / 'pool_job'
def cli(step, module, extra=(), data=True):
    if local_ok('pool_' + step):
        stamp(f'pool_{step} đã có — bỏ qua'); return
    args = [PY, '-m', 'c4.cli.' + module, *extra, '--job-dir', POOL_JOB] + (['--data-root', POOL_ROOT] if data else [])
    command('pool_' + step, args, quiet=True)
    mark_local('pool_' + step)
cli('index', 'lidar_index')
command('pool_split_check', [PY, SCRIPT, 'split-manifest', '--job', POOL_JOB])   # T=1389,V=3973,S=279,P=1112 + sha chia
cli('tier0', 'lidar_t0')
cli('clip', 'extract_clip', ['--device', 'cuda'])
cli('context_cv', 'context', ['cv'])
cli('context_fit', 'context', ['fit', '--split', 'V'])
cli('context_run', 'context', ['run'])
need = ['lidar/index.parquet', 'lidar/z0.npy', 'lidar/filter.parquet', 'split_manifest.csv', 'cache/context_frame.parquet']
miss = [n for n in need if not (POOL_JOB / n).is_file()]
assert not miss, f'job pool thiếu {miss}'
done('pool_prepared', job=str(POOL_JOB))
stamp('POOL JOB OK')

In [ ]:
# @title 9 · Cấu hình A (mốc): rare_v24 lidar_only, cluster_only, pool V / ref V — PHẢI chạy trước khi có t1/
assert 'POOL_JOB' in globals(), 'Chạy ô 8 trước'
ENGINES = POOL_JOB / 'engines' / 'rare_v24'

def restore_run(name):
    sid = (SAVE / 'runs' / name / 'sid.txt').read_text().strip()
    dst = ENGINES / sid
    if not (dst / 'rare_objects.parquet').is_file():
        shutil.copytree(SAVE / 'runs' / name / 'sel', dst, dirs_exist_ok=True)
    return sid

def rare_run(name, profile, expect_mode):
    if is_done(name):
        sid = restore_run(name); stamp(f'{name} đã xong (sid {sid}) — bỏ qua'); return sid
    tail = command(name, [PY, '-m', 'c4.cli.rare', 'run', '--job-dir', POOL_JOB, '--data-root', POOL_ROOT,
                          '--profile', profile, '--pool-split', 'V', '--ref-split', 'V'])
    sel = Path(re.findall(r'-> (\S+rare_v24\S+)\s*$', tail, re.M)[-1])
    cfg = json.loads((sel / 'config_used.json').read_text(encoding='utf-8'))
    assert (cfg['modelMode'], cfg['profile'], cfg['pool_split'], cfg['ref_split']) == (expect_mode, profile, 'V', 'V'), cfg
    shutil.copytree(sel, SAVE / 'runs' / name / 'sel', dirs_exist_ok=True)
    (SAVE / 'runs' / name / 'sid.txt').write_text(sel.name)
    done(name, sid=sel.name, modelMode=cfg['modelMode'], profile=profile, warnings=cfg['warnings'],
         rare_def_hash=cfg['rare_def_hash'], candidates_cfg_sha256=cfg['candidates_cfg_sha256'])
    return sel.name

def crops(name, sid):
    # cache crops nằm ở job LOCAL ⇒ đánh dấu local (runtime mới phải crops lại nếu cấu hình sau chưa xong)
    if local_ok(name):
        stamp(f'{name} đã có trong runtime này — bỏ qua'); return
    command(name, [PY, '-m', 'c4.cli.rare', 'crops', '--job-dir', POOL_JOB, '--data-root', POOL_ROOT,
                   '--device', 'cuda', '--sid', sid])
    mark_local(name); done(name, sid=sid)

if not is_done('A_cluster_lidar'):
    assert not (POOL_JOB / 't1' / 'preds_orig.pkl').exists(), 'đã có t1/ ⇒ A sẽ không còn là cluster_only'
SID_A = rare_run('A_cluster_lidar', 'lidar_only', 'cluster_only')
if RUN_A2_MULTI_CLUSTER:
    if not is_done('A2_cluster_multi'):
        assert not (POOL_JOB / 't1' / 'preds_orig.pkl').exists(), 'đã có t1/ ⇒ A2 không còn là cluster_only'
    if not is_done('A2_cluster_multi'):
        crops('crops_A', SID_A)
    SID_A2 = rare_run('A2_cluster_multi', 'multisensor', 'cluster_only')
stamp('CẤU HÌNH A OK')

In [ ]:
# @title 10 · Chờ train xong (nhịp 60 s: epoch, ETA, VRAM) → lưu checkpoint + cấu hình + sha256 lên Drive
assert 'SID_A' in globals() and 'SEED_FILES' in globals(), 'Chạy ô 7 và 9 trước'
def epochs_done():
    eps = [int(m.group(1)) for f in PCDET_OUT.rglob('checkpoint_epoch_*.pth')
           if (m := re.search(r'checkpoint_epoch_(\d+)\.pth$', f.name))] if PCDET_OUT.is_dir() else []
    return max(eps) if eps else 0

def heartbeat():
    k = epochs_done()
    el = time.time() - TRAIN_T0
    eta = (el / k * (EPOCHS - k)) if k else None
    tail = TRAIN_LOG.read_text(encoding='utf-8', errors='replace')[-600:].replace('\r', '\n').strip().splitlines()
    stamp(f"epoch {k}/{EPOCHS} · {el/60:.0f} phút · ETA {'?' if eta is None else f'{eta/60:.0f} phút'} · "
          f"VRAM {gpu_used_mib()} MiB · {tail[-1][:160] if tail else ''}")

if not seed_on_drive():
    t_wait = time.time()
    while True:
        while (TRAIN_PROC is not None and TRAIN_PROC.poll() is None) or (TRAIN_PROC is None and train_running()):
            heartbeat(); time.sleep(60)
        rc = TRAIN_PROC.returncode if TRAIN_PROC is not None else (0 if (EXP / 't1/ckpt/seed_latest.pth').is_file() else 1)
        log_tail = TRAIN_LOG.read_text(encoding='utf-8', errors='replace')[-6000:]
        if rc == 3 and 'out of memory' in log_tail.lower() and TRAIN_BATCH > 4:
            mirror_ckpts(final=True)
            stamp(f'OOM ở batch {TRAIN_BATCH} ⇒ tự khởi động lại với batch {TRAIN_BATCH // 2} (resume từ ckpt nếu có)')
            start_train(TRAIN_BATCH // 2)
            continue
        break
    mirror_ckpts(final=True)
    TIMING.append(dict(step='train_seed', elapsed_s=round(time.time() - t_wait, 1), exit_code=rc, batch=TRAIN_BATCH,
                       epochs=EPOCHS, note='thời gian chờ ở ô 10; tổng train = timing train.log'))
    save_timing()
    if rc != 0 or not (EXP / 't1/ckpt/seed_latest.pth').is_file():
        print(log_tail)
        raise RuntimeError(f'train_seed exit {rc} — xem {SAVE / "logs/train.log"}; ckpt giữa chừng ở {TRAIN_SAVE}')
    for rel, d in SEED_FILES.items():
        to_drive(EXP / 't1' / rel, SEED_SAVE / d)
SYNC_STOP.set()
TRAIN_CFG = (SEED_SAVE / 'train_config.yaml').read_text(encoding='utf-8')
print(TRAIN_CFG)
assert 'gt_sampling: false' in TRAIN_CFG and f'sweeps: {SWEEPS}' in TRAIN_CFG
CKPT_SHA = (SEED_SAVE / 'seed_latest.pth.sha256').read_text().split()[0]
done('train', ckpt_sha256=CKPT_SHA, epochs=EPOCHS)
stamp(f'SEED OK · ckpt sha256 {CKPT_SHA}')

In [ ]:
# @title 11 · Suy luận PointPillars trên MỌI frame job pool (2 lượt: gốc + lật) → t1/ ; không đọc nhãn, không eval
assert 'CKPT_SHA' in globals(), 'Chạy ô 10 trước'
T1, T1_SAVE = POOL_JOB / 't1', SAVE / 't1'
T1_FILES = ['preds_orig.pkl', 'preds_flip.pkl', 'z1.npy', 'signals.parquet', 'signals.parquet.manifest.json']
if all(verified(T1_SAVE / n) for n in T1_FILES) and (T1_SAVE / 'ckpt_sha256.txt').read_text().strip() == CKPT_SHA:
    T1.mkdir(parents=True, exist_ok=True)
    for n in T1_FILES:
        if not (T1 / n).is_file():
            shutil.copy2(T1_SAVE / n, T1 / n)
    stamp('t1/ của đúng checkpoint đã có trên Drive — khôi phục, bỏ suy luận')
else:
    restore_seed()
    command('pp_infer', [PY, '-m', 'c4.lidar.tier1.infer_t1', '--exp', EXP, '--job', POOL_JOB, '--nusc', POOL_ROOT,
                         '--batch', 8])
    for n in T1_FILES:
        to_drive(T1 / n, T1_SAVE / n)
    (T1_SAVE / 'ckpt_sha256.txt').write_text(CKPT_SHA)
done('pp_infer', ckpt_sha256=CKPT_SHA)
stamp('PP INFER OK')

In [ ]:
# @title 12 · Cấu hình B (cluster+pp lidar_only) và C (cluster+pp multisensor: crops OpenCLIP rồi chạy)
assert (POOL_JOB / 't1' / 'preds_orig.pkl').is_file(), 'Chạy ô 11 trước'
SID_B = rare_run('B_pp_lidar', 'lidar_only', 'cluster+pp_frozen')
if not is_done('C_pp_multi'):
    crops('crops_B', SID_B)
SID_C = rare_run('C_pp_multi', 'multisensor', 'cluster+pp_frozen')
stamp(f'CẤU HÌNH B/C OK · B={SID_B} · C={SID_C}')

In [ ]:
# @title 13 · Chẩn đoán trên V (KHÔNG phải T): IY@B capture B, reach r=1 m, bootstrap cụm scene 1000 lượt seed 20261013
assert 'SID_C' in globals(), 'Chạy ô 12 trước'
runs = ['A_cluster_lidar', 'B_pp_lidar', 'C_pp_multi'] + (['A2_cluster_multi'] if RUN_A2_MULTI_CLUSTER else [])
pairs = ['B_pp_lidar-A_cluster_lidar', 'C_pp_multi-A_cluster_lidar', 'C_pp_multi-B_pp_lidar',
         'A_cluster_lidar-M1_legacy', 'B_pp_lidar-M1_legacy']
if RUN_A2_MULTI_CLUSTER:
    pairs.append('C_pp_multi-A2_cluster_multi')
args = [PY, SCRIPT, 'diag', '--job', POOL_JOB, '--data-root', POOL_ROOT, '--ckpt', SEED_SAVE / 'seed_latest.pth',
        '--out', SAVE / 'diag']
for r in runs:
    args += ['--run', f"{r}={SAVE / 'runs' / r / 'sel'}"]
for p in pairs:
    args += ['--pair', p]
if is_done('diag') and (SAVE / 'diag/diag_report.json').is_file():
    stamp('diag đã xong — in lại bảng')
    print((SAVE / 'diag/diag_table.md').read_text(encoding='utf-8'))
else:
    command('diag', args)
    done('diag', report_sha256=digest(SAVE / 'diag/diag_report.json'))

In [ ]:
# @title 14 · Đóng gói kết quả: SHA256SUMS + results_manifest.json trên Drive; in tổng thời gian
assert is_done('diag'), 'Chạy ô 13 trước'
def sha_all():
    lines = []
    for f in sorted(SAVE.rglob('*')):
        if f.is_file() and not f.name.endswith(('.part', '.sha256')) and f.name not in ('SHA256SUMS.txt',):
            lines.append(f'{digest(f)}  {f.relative_to(SAVE).as_posix()}')
    (SAVE / 'SHA256SUMS.txt').write_text('\n'.join(lines) + '\n', encoding='utf-8')
    return len(lines)
n = measured('sha_all', sha_all)
rep = json.loads((SAVE / 'diag/diag_report.json').read_text(encoding='utf-8'))
manifest = dict(run_id=RUN_ID, repo_sha=REPO_SHA, gpu=GPU, params=dict(epochs=EPOCHS, batch=TRAIN_BATCH if 'TRAIN_BATCH' in globals() else BATCH,
                workers=WORKERS, sweeps=SWEEPS, a2=RUN_A2_MULTI_CLUSTER), seed_split=SEED_SPLIT, ckpt_sha256=CKPT_SHA,
                diag=dict(id=rep['diag_id'], label=rep['label'], B=rep['B'], targets=rep['n_target_instances'],
                          methods={k: {kk: v.get(kk) for kk in ('iy', 'ci95', 'reach', 'reach_ci95', 'modelMode')}
                                   for k, v in rep['methods'].items()}, deltas=rep['deltas']),
                pool_manifest=dict(dir=str(POOL_DIR), parts=len(POOL_MANIFEST['parts'])),
                seed_manifest=dict(dir=str(SEED_DIR), scenes_sha256=SEED_MANIFEST['seed_scenes_sha256']),
                files_hashed=n, T_opened=False, steps=TIMING)
(SAVE / 'results_manifest.json').write_text(json.dumps(manifest, indent=2, default=str), encoding='utf-8')
print((SAVE / 'diag/diag_table.md').read_text(encoding='utf-8'))
size = sum(f.stat().st_size for f in SAVE.rglob('*') if f.is_file()) / 1e9
stamp(f'XONG · {n} file có sha256 · {size:.2f} GB ở {SAVE} · tổng notebook {(time.time() - T_NOTEBOOK)/3600:.2f} h (phiên này)')